In [ ]:
cd /content/drive/MyDrive/learning_work/churn_prediction/assets

/content/drive/MyDrive/learning_work/churn_prediction/assets


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

## Read data

In [ ]:
demo = pd.read_csv("demo.csv")
card_info = pd.read_csv("card_info.csv")
cc_txn = pd.read_csv("cc_txn.csv")
sa_bal = pd.read_csv("sa_bal.csv")
dtxn = pd.read_csv("dtxn.csv")
y_train = pd.read_csv("y_train.csv")

print(demo.shape)
print(card_info.shape)
print(cc_txn.shape)
print(sa_bal.shape)
print(dtxn.shape)
print(y_train.shape)

(52762, 7)
(60296, 4)
(3223075, 5)
(633144, 3)
(490599, 4)
(494, 2)


In [ ]:
last_date = pd.to_datetime("2017-12-31")

## Manage credit card transaction



> **To create year_month, month, and quarter from txn_dt**

In [ ]:
cc_txn["txn_dt"] = pd.to_datetime(cc_txn["txn_dt"], errors="coerce")
cc_txn["year_month"] = cc_txn["txn_dt"].dt.to_period("M").astype(str)
cc_txn["month"] = cc_txn["txn_dt"].dt.month
cc_txn["quarter"] = "Q" + ((cc_txn["month"] - 1) // 3 + 1).astype(str)
cc_txn.head()

,mcc,txn_dt,user_id,txn_amt,card_no,year_month,month,quarter
0,mcc_cat10,2017-04-12,17863,16391.0,21616.0,2017-04,4,Q2
1,mcc_cat11,2017-07-12,7682,117200.0,9438.0,2017-07,7,Q3
2,mcc_cat12,2017-05-23,33603,55602.0,39476.0,2017-05,5,Q2
3,mcc_cat4,2017-11-03,44040,38894.0,50979.0,2017-11,11,Q4
4,mcc_cat14,2017-12-18,4529,29524.0,5582.0,2017-12,12,Q4




> **To check credit card's most recent usage date of each user_id**





In [ ]:
last_txn = cc_txn.groupby(["user_id"], as_index=False).agg(last_txn_dt=("txn_dt", "max"))
last_txn["days_since_last_txn"] = (last_date - last_txn["last_txn_dt"]).dt.days
last_txn.head()

,user_id,last_txn_dt,days_since_last_txn
0,0,2017-12-28,3
1,1,2017-12-31,0
2,2,2017-12-21,10
3,3,2017-12-25,6
4,4,2017-12-31,0




> **To view total tranaction amount, average tranaction amount, utilization_ratio and number of transaction in each quater of each user_id**





In [ ]:
card_agg = pd.DataFrame(cc_txn.groupby(['user_id', 'quarter', 'card_no']).agg(sum_amt=('txn_amt', 'sum'), avg_amt=('txn_amt', 'mean'), cnt_trans=("user_id","count"))).reset_index()
card_agg = card_agg.merge(card_info, on=['user_id', 'card_no'], how='left')
card_agg['utilization_ratio'] = (card_agg['sum_amt']) / card_agg['cr_lmt_amt'] *100

card_count = card_agg.groupby(["user_id"], as_index=False).agg(total_cnt_card = ("card_no", "nunique"))
card_agg1 = card_agg.groupby(["user_id", "quarter"], as_index=False).agg(sum_amt=("sum_amt", "sum"), avg_amt=("avg_amt", "mean"),
                                                                       cnt_trans=("cnt_trans", "sum"), utilization_ratio=("utilization_ratio", "mean"),
                                                                       cnt_card = ("card_no", "count"))

features = ["sum_amt","avg_amt","cnt_trans","utilization_ratio", "cnt_card"]
card_piv = card_agg1.pivot(index=["user_id"], columns="quarter", values=features)
card_piv.columns = [f"{feature}_{month}" for feature, month in card_piv.columns]
card_piv = card_piv.reset_index()
card_piv = card_piv.fillna(0)

card_piv = card_piv.merge(card_count, on = ['user_id'], how='left')
card_piv = card_piv.merge(last_txn[['user_id', 'days_since_last_txn']], on = ['user_id'], how='left')
card_piv["days_since_last_txn"] = card_piv["days_since_last_txn"].fillna(0)
card_piv.head()

,user_id,sum_amt_Q1,sum_amt_Q2,sum_amt_Q3,sum_amt_Q4,avg_amt_Q1,avg_amt_Q2,avg_amt_Q3,avg_amt_Q4,cnt_trans_Q1,...,utilization_ratio_Q1,utilization_ratio_Q2,utilization_ratio_Q3,utilization_ratio_Q4,cnt_card_Q1,cnt_card_Q2,cnt_card_Q3,cnt_card_Q4,total_cnt_card,days_since_last_txn
0,0,254900.0,638357.0,348156.0,504694.0,19607.692308,39897.312500,29013.000000,33646.266667,13.0,...,104.467213,261.621721,142.686885,206.841803,1.0,1.0,1.0,1.0,1,3
1,1,518002.0,495560.0,436762.0,784067.0,39846.307692,35397.142857,36396.833333,31362.680000,13.0,...,143.490859,137.274238,120.986704,217.193075,1.0,1.0,1.0,1.0,1,0
2,2,110381.0,706560.0,373447.0,442390.0,27595.250000,44160.000000,33949.727273,36865.833333,4.0,...,28.670390,183.522078,96.999221,114.906494,1.0,1.0,1.0,1.0,1,10
3,3,664662.0,304854.0,940155.0,974785.0,94951.714286,50809.000000,62677.000000,57340.294118,7.0,...,128.313127,58.852124,181.497104,188.182432,1.0,1.0,1.0,1.0,1,6
4,4,790782.0,855577.0,553621.0,773503.0,65898.500000,85557.700000,55362.100000,70318.454545,12.0,...,88.951856,96.240382,62.274578,87.008211,1.0,1.0,1.0,1.0,1,0




> **To compare the difference of total tranaction, number of transaction, and utilization ratio between the recent 3 months (Q4) and the previous 3 months (Q3)**



In [ ]:
# สมมติ card_month มี user_id, mm, sum_amt, cnt_trans, utilization_ratio
# เทียบ 3 เดือนล่าสุดกับ 3 เดือนก่อนหน้า
recent_3m = card_agg1[card_agg1["quarter"] == 'Q4']
prev_3m = card_agg1[card_agg1["quarter"] == 'Q3']

recent_feat = recent_3m.groupby("user_id", as_index=False).agg(sum_amt_recent_3m=("sum_amt", "sum"),
                                                               cnt_trans_recent_3m=("cnt_trans", "sum"),
                                                               util_recent_3m=("utilization_ratio", "mean"))


prev_feat = prev_3m.groupby("user_id", as_index=False).agg(sum_amt_prev_3m=("sum_amt", "sum"),
                                                           cnt_trans_prev_3m=("cnt_trans", "sum"),
                                                           util_prev_3m=("utilization_ratio", "mean"))

trend_feat = recent_feat.merge(prev_feat, on="user_id", how="left")

trend_feat["sum_amt_3m_diff"] = trend_feat["sum_amt_recent_3m"] - trend_feat["sum_amt_prev_3m"]
trend_feat["cnt_trans_3m_diff"] = trend_feat["cnt_trans_recent_3m"] - trend_feat["cnt_trans_prev_3m"]
trend_feat["sum_amt_3m_ratio"] = trend_feat["sum_amt_recent_3m"] / trend_feat["sum_amt_prev_3m"].replace(0, np.nan)
trend_feat["cnt_trans_3m_ratio"] = trend_feat["cnt_trans_recent_3m"] / trend_feat["cnt_trans_prev_3m"].replace(0, np.nan)
trend_feat.head()

,user_id,sum_amt_recent_3m,cnt_trans_recent_3m,util_recent_3m,sum_amt_prev_3m,cnt_trans_prev_3m,util_prev_3m,sum_amt_3m_diff,cnt_trans_3m_diff,sum_amt_3m_ratio,cnt_trans_3m_ratio
0,0,504694.0,15,206.841803,348156.0,12,142.686885,156538.0,3,1.449620,1.250000
1,1,784067.0,25,217.193075,436762.0,12,120.986704,347305.0,13,1.795181,2.083333
2,2,442390.0,12,114.906494,373447.0,11,96.999221,68943.0,1,1.184613,1.090909
3,3,974785.0,17,188.182432,940155.0,15,181.497104,34630.0,2,1.036834,1.133333
4,4,773503.0,11,87.008211,553621.0,10,62.274578,219882.0,1,1.397171,1.100000




> **To view the total and ratio of transaction amount in each merchant category**



In [ ]:
mcc_feat = cc_txn.groupby("user_id", as_index=False).agg(n_mcc=("mcc", "nunique"), total_txn=("mcc", "count"))
mcc_amt = cc_txn.groupby(["user_id", "mcc"], as_index=False).agg(mcc_sum_amt=("txn_amt", "sum"))
mcc_pivot = mcc_amt.pivot_table(index="user_id",columns="mcc",values="mcc_sum_amt",
                                aggfunc="sum",fill_value=0)

mcc_pivot.columns = [f"mcc_{col}_sum_amt" for col in mcc_pivot.columns]

mcc_pivot = mcc_pivot.reset_index()

mcc_cols = [col for col in mcc_pivot.columns if col.startswith("mcc_")]

mcc_pivot["mcc_total_amt"] = mcc_pivot[mcc_cols].sum(axis=1)

for col in mcc_cols:
    mcc_pivot[col.replace("_sum_amt", "_ratio")] = (mcc_pivot[col] / mcc_pivot["mcc_total_amt"].replace(0, np.nan))

# ลูกค้าที่กำลัง churn อาจมี category diversity ลดลง หรือเหลือใช้แค่บางหมวด
mcc_pivot

,user_id,mcc_mcc_cat1_sum_amt,mcc_mcc_cat10_sum_amt,mcc_mcc_cat11_sum_amt,mcc_mcc_cat12_sum_amt,mcc_mcc_cat13_sum_amt,mcc_mcc_cat14_sum_amt,mcc_mcc_cat15_sum_amt,mcc_mcc_cat16_sum_amt,mcc_mcc_cat2_sum_amt,...,mcc_mcc_cat15_ratio,mcc_mcc_cat16_ratio,mcc_mcc_cat2_ratio,mcc_mcc_cat3_ratio,mcc_mcc_cat4_ratio,mcc_mcc_cat5_ratio,mcc_mcc_cat6_ratio,mcc_mcc_cat7_ratio,mcc_mcc_cat8_ratio,mcc_mcc_cat9_ratio
0,0,265794.0,149038.0,26189.0,140380.0,51348.0,0.0,196999.0,95693.0,50821.0,...,0.112822,0.054804,0.029105,0.082126,0.065978,0.110046,0.093774,0.067545,0.000000,0.021424
1,1,145315.0,59052.0,63103.0,142684.0,183189.0,28863.0,349915.0,70302.0,2862.0,...,0.156604,0.031464,0.001281,0.008264,0.084936,0.064008,0.097389,0.106347,0.071199,0.100040
2,2,215599.0,51336.0,7269.0,122175.0,48199.0,91520.0,64748.0,57683.0,236499.0,...,0.039655,0.035328,0.144845,0.048812,0.125889,0.105980,0.057599,0.042559,0.034032,0.036967
3,3,129301.0,91061.0,62939.0,283892.0,51292.0,185176.0,201412.0,359093.0,89460.0,...,0.069827,0.124492,0.031015,0.042494,0.190933,0.033794,0.076927,0.009341,0.132796,0.009764
4,4,190764.0,86141.0,233295.0,77245.0,29812.0,552109.0,644112.0,15571.0,140544.0,...,0.216619,0.005237,0.047266,0.029724,0.046550,0.066681,0.079199,0.000000,0.027245,0.088214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52757,52757,67664.0,70780.0,0.0,77627.0,178669.0,102263.0,354855.0,121204.0,0.0,...,0.151667,0.051803,0.000000,0.071704,0.041155,0.035410,0.020474,0.175026,0.089976,0.150362
52758,52758,309536.0,224617.0,468679.0,127956.0,380851.0,156778.0,0.0,23529.0,391193.0,...,0.000000,0.006755,0.112313,0.125534,0.036650,0.036976,0.063161,0.053172,0.047807,0.038625
52759,52759,328679.0,268195.0,142214.0,609343.0,172061.0,63012.0,232505.0,92522.0,201017.0,...,0.060728,0.024166,0.052503,0.073247,0.000000,0.064717,0.093767,0.049168,0.106942,0.061169
52760,52760,340994.0,441058.0,387125.0,435122.0,127414.0,624046.0,61974.0,442428.0,275753.0,...,0.013724,0.097972,0.061063,0.086916,0.037621,0.032107,0.030968,0.003839,0.023973,0.090156


In [ ]:
card_piv = card_piv.merge(trend_feat, on = ['user_id'], how='left')
card_piv = card_piv.merge(mcc_pivot, on = ['user_id'], how='left')
card_piv.head()

,user_id,sum_amt_Q1,sum_amt_Q2,sum_amt_Q3,sum_amt_Q4,avg_amt_Q1,avg_amt_Q2,avg_amt_Q3,avg_amt_Q4,cnt_trans_Q1,...,mcc_mcc_cat15_ratio,mcc_mcc_cat16_ratio,mcc_mcc_cat2_ratio,mcc_mcc_cat3_ratio,mcc_mcc_cat4_ratio,mcc_mcc_cat5_ratio,mcc_mcc_cat6_ratio,mcc_mcc_cat7_ratio,mcc_mcc_cat8_ratio,mcc_mcc_cat9_ratio
0,0,254900.0,638357.0,348156.0,504694.0,19607.692308,39897.312500,29013.000000,33646.266667,13.0,...,0.112822,0.054804,0.029105,0.082126,0.065978,0.110046,0.093774,0.067545,0.000000,0.021424
1,1,518002.0,495560.0,436762.0,784067.0,39846.307692,35397.142857,36396.833333,31362.680000,13.0,...,0.156604,0.031464,0.001281,0.008264,0.084936,0.064008,0.097389,0.106347,0.071199,0.100040
2,2,110381.0,706560.0,373447.0,442390.0,27595.250000,44160.000000,33949.727273,36865.833333,4.0,...,0.039655,0.035328,0.144845,0.048812,0.125889,0.105980,0.057599,0.042559,0.034032,0.036967
3,3,664662.0,304854.0,940155.0,974785.0,94951.714286,50809.000000,62677.000000,57340.294118,7.0,...,0.069827,0.124492,0.031015,0.042494,0.190933,0.033794,0.076927,0.009341,0.132796,0.009764
4,4,790782.0,855577.0,553621.0,773503.0,65898.500000,85557.700000,55362.100000,70318.454545,12.0,...,0.216619,0.005237,0.047266,0.029724,0.046550,0.066681,0.079199,0.000000,0.027245,0.088214


## Manage Amount of money inbound and outbound



> **Create quater from month and net cashflow from amount inbound and outbound in each quarter**

In [ ]:
dtxn['net_flow'] = dtxn['amt_in'] - dtxn['amt_out']
dtxn["quarter"] = "Q" + ((dtxn["mm"] - 1) // 3 + 1).astype(str)

# Aggregate net_flow before pivoting to ensure unique combinations of user_id and quarter
dtxn_agg = dtxn.groupby(['user_id', 'quarter']).agg(net_flow=('net_flow', 'sum')).reset_index()

features = ["net_flow"]
dtxn_piv = dtxn_agg.pivot(index=["user_id"], columns="quarter", values=features)
dtxn_piv.columns = [f"{feature}_{quarter}" for feature, quarter in dtxn_piv.columns]
dtxn_piv = dtxn_piv.reset_index()
dtxn_piv = dtxn_piv.fillna(0)
dtxn_piv.head()

,user_id,net_flow_Q1,net_flow_Q2,net_flow_Q3,net_flow_Q4
0,0,82654.0,-77909.0,-34348.0,0.0
1,1,-64395.0,29948.0,-66704.0,23453.0
2,2,-32230.0,19242.0,205244.0,22388.0
3,3,33876.0,-133234.0,271985.0,-9642.0
4,4,-147192.0,-32203.0,-204052.0,-292424.0




> **To find sum, average and max amount of money inbound and outbound of each user**


> **To calculate net cashflow and out/in ratio**





In [ ]:
dtxn_feat = dtxn.groupby("user_id", as_index=False).agg(total_amt_in=("amt_in", "sum"), total_amt_out=("amt_out", "sum"),
                                                        avg_amt_in=("amt_in", "mean"), avg_amt_out=("amt_out", "mean"),
                                                        max_amt_in=("amt_in", "max"), max_amt_out=("amt_out", "max"))
dtxn_feat["net_cashflow"] = dtxn_feat["total_amt_in"] - dtxn_feat["total_amt_out"]
dtxn_feat["out_in_ratio"] = dtxn_feat["total_amt_out"] / dtxn_feat["total_amt_in"].replace(0, np.nan)
dtxn_feat

,user_id,total_amt_in,total_amt_out,avg_amt_in,avg_amt_out,max_amt_in,max_amt_out,net_cashflow,out_in_ratio
0,0,147652.0,177255.0,18456.500000,22156.875000,86274.0,84557.0,-29603.0,1.200492
1,1,131178.0,208876.0,13117.800000,20887.600000,38031.0,51018.0,-77698.0,1.592310
2,2,387124.0,172480.0,43013.777778,19164.444444,112077.0,73924.0,214644.0,0.445542
3,3,569276.0,406291.0,51752.363636,36935.545455,127230.0,136672.0,162985.0,0.713698
4,4,191290.0,867161.0,21254.444444,96351.222222,101097.0,167753.0,-675871.0,4.533227
...,...,...,...,...,...,...,...,...,...
52757,52757,364472.0,485480.0,36447.200000,48548.000000,163322.0,173300.0,-121008.0,1.332009
52758,52758,509097.0,468741.0,50909.700000,46874.100000,198976.0,112396.0,40356.0,0.920730
52759,52759,250124.0,378732.0,27791.555556,42081.333333,142284.0,123970.0,-128608.0,1.514177
52760,52760,136002.0,219277.0,19428.857143,31325.285714,99837.0,108125.0,-83275.0,1.612307


In [ ]:
dtxn_piv = dtxn_piv.merge(dtxn_feat, on = ['user_id'], how='left')
dtxn_piv.head()

,user_id,net_flow_Q1,net_flow_Q2,net_flow_Q3,net_flow_Q4,total_amt_in,total_amt_out,avg_amt_in,avg_amt_out,max_amt_in,max_amt_out,net_cashflow,out_in_ratio
0,0,82654.0,-77909.0,-34348.0,0.0,147652.0,177255.0,18456.500000,22156.875000,86274.0,84557.0,-29603.0,1.200492
1,1,-64395.0,29948.0,-66704.0,23453.0,131178.0,208876.0,13117.800000,20887.600000,38031.0,51018.0,-77698.0,1.592310
2,2,-32230.0,19242.0,205244.0,22388.0,387124.0,172480.0,43013.777778,19164.444444,112077.0,73924.0,214644.0,0.445542
3,3,33876.0,-133234.0,271985.0,-9642.0,569276.0,406291.0,51752.363636,36935.545455,127230.0,136672.0,162985.0,0.713698
4,4,-147192.0,-32203.0,-204052.0,-292424.0,191290.0,867161.0,21254.444444,96351.222222,101097.0,167753.0,-675871.0,4.533227


## Manage saving account balance



> **To view maximum saving account balance in each quarter**



In [ ]:
features = ["max_sa_bal"]
sa_bal["quarter"] = "Q" + ((sa_bal["mm"] - 1) // 3 + 1).astype(str)

# Aggregate max_sa_bal before pivoting to handle duplicate entries
sa_bal_agg = sa_bal.groupby(['user_id', 'quarter']).agg(max_sa_bal=('max_sa_bal', 'max')).reset_index()

sa_bal_piv = sa_bal_agg.pivot(index=["user_id"], columns="quarter", values=features)
sa_bal_piv.columns = [f"{feature}_{quarter}" for feature, quarter in sa_bal_piv.columns]
sa_bal_piv = sa_bal_piv.reset_index()
sa_bal_piv.head()

,user_id,max_sa_bal_Q1,max_sa_bal_Q2,max_sa_bal_Q3,max_sa_bal_Q4
0,0,646838.0,197374.0,987037.0,345973.0
1,1,939070.0,784543.0,567590.0,206761.0
2,2,726292.0,487902.0,1232344.0,280998.0
3,3,822564.0,719043.0,1058860.0,923919.0
4,4,771838.0,1108262.0,1379966.0,275523.0




> **To compare the difference of average and maximum of maximum saving account balance between the recent 3 months (Q4) and the previous 3 months (Q3)**



In [ ]:
sa_recent = sa_bal[sa_bal["quarter"] == 'Q4']
sa_prev = sa_bal[sa_bal["quarter"] == 'Q3']

sa_recent_feat = sa_recent.groupby("user_id", as_index=False).agg(sa_bal_recent_3m_avg=("max_sa_bal", "mean"),
                                                                  sa_bal_recent_3m_max=("max_sa_bal", "max"))

sa_prev_feat = sa_prev.groupby("user_id", as_index=False).agg(sa_bal_prev_3m_avg=("max_sa_bal", "mean"),
                                                              sa_bal_prev_3m_max=("max_sa_bal", "max"))

sa_trend = sa_recent_feat.merge(sa_prev_feat, on="user_id", how="left")
sa_trend["sa_bal_3m_avg_diff"] = sa_trend["sa_bal_recent_3m_avg"] - sa_trend["sa_bal_prev_3m_avg"]
sa_trend["sa_bal_3m_avg_ratio"] = sa_trend["sa_bal_recent_3m_avg"]/sa_trend["sa_bal_prev_3m_avg"].replace(0, np.nan)
sa_trend.head()

,user_id,sa_bal_recent_3m_avg,sa_bal_recent_3m_max,sa_bal_prev_3m_avg,sa_bal_prev_3m_max,sa_bal_3m_avg_diff,sa_bal_3m_avg_ratio
0,0,169167.666667,345973.0,529284.666667,987037.0,-360117.000000,0.319616
1,1,121790.666667,206761.0,269846.000000,567590.0,-148055.333333,0.451334
2,2,163973.333333,280998.0,761011.333333,1232344.0,-597038.000000,0.215468
3,3,793793.333333,923919.0,663049.333333,1058860.0,130744.000000,1.197186
4,4,105840.666667,275523.0,804021.333333,1379966.0,-698180.666667,0.131639


In [ ]:
sa_bal_piv = sa_bal_piv.merge(sa_trend, on = ['user_id'], how='left')
sa_bal_piv.head()

,user_id,max_sa_bal_Q1,max_sa_bal_Q2,max_sa_bal_Q3,max_sa_bal_Q4,sa_bal_recent_3m_avg,sa_bal_recent_3m_max,sa_bal_prev_3m_avg,sa_bal_prev_3m_max,sa_bal_3m_avg_diff,sa_bal_3m_avg_ratio
0,0,646838.0,197374.0,987037.0,345973.0,169167.666667,345973.0,529284.666667,987037.0,-360117.000000,0.319616
1,1,939070.0,784543.0,567590.0,206761.0,121790.666667,206761.0,269846.000000,567590.0,-148055.333333,0.451334
2,2,726292.0,487902.0,1232344.0,280998.0,163973.333333,280998.0,761011.333333,1232344.0,-597038.000000,0.215468
3,3,822564.0,719043.0,1058860.0,923919.0,793793.333333,923919.0,663049.333333,1058860.0,130744.000000,1.197186
4,4,771838.0,1108262.0,1379966.0,275523.0,105840.666667,275523.0,804021.333333,1379966.0,-698180.666667,0.131639


## Join data



> **After joining overall data, we calculate age of user from birth_year and account aging from account_start_date**




In [ ]:
data = demo.merge(y_train, on=['user_id'], how='right')
data = data.merge(card_piv, on=['user_id'], how='left')
data = data.merge(sa_bal_piv, on=['user_id'], how='left')
data["account_start_date"] = pd.to_datetime(data["account_start_date"], errors="coerce", dayfirst=True)
data['account_aging_month'] = ((last_date.year - data["account_start_date"].dt.year) * 12 + (last_date.month - data["account_start_date"].dt.month))
data['user_age'] = 2017 - data['birth_year']
data.drop(columns=['account_start_date', 'birth_year', 'family_income_segment_code'], inplace=True)
print(data.shape)
data.head()

(494, 82)


,user_id,gender,marital_status,individual_income_segment_code,label,sum_amt_Q1,sum_amt_Q2,sum_amt_Q3,sum_amt_Q4,avg_amt_Q1,...,max_sa_bal_Q3,max_sa_bal_Q4,sa_bal_recent_3m_avg,sa_bal_recent_3m_max,sa_bal_prev_3m_avg,sa_bal_prev_3m_max,sa_bal_3m_avg_diff,sa_bal_3m_avg_ratio,account_aging_month,user_age
0,2723,F,2,15,1.0,564528.0,1012963.0,674978.0,1237067.0,70566.000000,...,862413.0,629437.0,288810.000000,629437.0,4.437793e+05,862413.0,-154969.333333,0.650796,470,79.0
1,44088,M,,03,1.0,472247.0,1250633.0,1134008.0,1295428.0,42931.545455,...,2328823.0,1000964.0,520877.666667,1000964.0,1.295088e+06,2328823.0,-774210.666667,0.402195,385,65.0
2,3139,F,3,08,0.0,372719.0,603220.0,768056.0,488722.0,37271.900000,...,1099177.0,718667.0,507269.000000,718667.0,6.294283e+05,1099177.0,-122159.333333,0.805920,460,66.0
3,25596,M,1,03,0.0,642111.0,525513.0,746737.0,452694.0,40131.937500,...,516441.0,129100.0,43033.333333,129100.0,4.146177e+05,516441.0,-371584.333333,0.103790,581,79.0
4,15409,M,,14,0.0,464573.0,489361.0,626925.0,732213.0,30971.533333,...,717097.0,485440.0,161813.333333,485440.0,4.020730e+05,717097.0,-240259.666667,0.402448,523,70.0


In [ ]:
data.to_csv('train_data_V2.csv', index=False)